# M3.7 · Dropout, weight decay, early stopping

Sinir ağlarında aşırı uyumla savaşın üç standart aracı. Inverted dropout'u ve early stopping kararını elle uyguluyor, saf Python'da dropout'un ileri/ geri geçişini yazıyor, NumPy'da MC dropout ile belirsizlik tahminini tek tensörde yapıyor, PyTorch'ta `train()/eval()` davranışını taşıyan kendi Dropout modülünü ve en iyi ağırlıkları geri yükleyen bir eğitim döngüsü kuruyor ve en sonda T stokastik geçişi tek bir batched hesapla yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-7-regularization-nn.html](https://m-gonul.github.io/pytorch-lab/m3-7-regularization-nn.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_7_regularization_nn.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def fwd():\n        y, m = need(ns, \"dropout_forward\")([2, 4, 6, 8], 0.5, random.Random(0))\n        r = random.Random(0); want = [1 if r.random() >= 0.5 else 0 for _ in range(4)]\n        ensure(list(m) == want, f\"maske {m}, {want} bekleniyordu (eleman başına sırayla rng.random())\")\n        close(y, [v * k / 0.5 for v, k in zip([2, 4, 6, 8], want)])\n        y2, m2 = need(ns, \"dropout_forward\")([2, 4], 0.5, random.Random(0), train=False)\n        ensure(list(y2) == [2, 4] and m2 is None, \"eval'de kimlik ve mask None\")\n    R.test(\"dropout_forward: maske, ölçek, eval\", fwd)\n\n    def unb():\n        rng = random.Random(1)\n        n = 20000\n        y, _ = need(ns, \"dropout_forward\")([3.0] * n, 0.3, rng)\n        ensure(abs(sum(y) / n - 3.0) < 0.05, f\"ortalama {sum(y) / n:.3f}, 3 bekleniyordu\")\n        return f\"20 000 örnek: ortalama {sum(y) / n:.4f} (girdi 3.0)\"\n    R.test(\"beklenen değer korunuyor\", unb)\n    R.test(\"dropout_backward\", lambda: close(need(ns, \"dropout_backward\")([1.0, 1.0, 2.0], [1, 0, 1], 0.2), [1.25, 0, 2.5]))\n\n    def es():\n        f = need(ns, \"early_stopping\")\n        ensure(tuple(f([0.9, 0.7, 0.6, 0.62, 0.65, 0.61], 2)) == (2, 4), \"elle örnek (2, 4)\")\n        ensure(tuple(f([1.0, 0.9, 0.8], 3)) == (2, 2), \"hiç durmazsa stop = son indeks\")\n        ensure(tuple(f([1.0, 0.995, 0.99, 0.985], 2, min_delta=0.01)) == (0, 2), \"min_delta küçük iyileşmeleri saymamalı\")\n    R.test(\"early_stopping (min_delta dahil)\", es)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"dropout_np / mc_predict: döngü yok\", lambda: (max_loops(src, 0, func=\"dropout_np\"), max_loops(src, 0, func=\"mc_predict\")))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def dn():\n        X = rng.normal(size=(50, 40))\n        Y, m = need(ns, \"dropout_np\")(X, 0.4, np.random.default_rng(1))\n        want = np.random.default_rng(1).random(X.shape) >= 0.4\n        ensure(np.array_equal(np.asarray(m, bool), want), \"maske rng.random(X.shape) >= p olmalı\")\n        close(Y, X * want / 0.6)\n    R.test(\"dropout_np\", dn)\n\n    def mc():\n        W1, b1 = rng.normal(size=(3, 16)), rng.normal(size=16) * 0.1\n        W2, b2 = rng.normal(size=(16, 4)), rng.normal(size=4) * 0.1\n        X = rng.normal(size=(10, 3))\n        mean, std = need(ns, \"mc_predict\")(W1, b1, W2, b2, X, 0.5, 200, np.random.default_rng(5))\n        r = np.random.default_rng(5)\n        H = np.maximum(0, X @ W1 + b1)\n        M = r.random((200,) + H.shape) >= 0.5\n        P = []\n        for t in range(200):\n            s = (H * M[t] / 0.5) @ W2 + b2\n            e = np.exp(s - s.max(1, keepdims=True)); P.append(e / e.sum(1, keepdims=True))\n        P = np.array(P)\n        close(mean, P.mean(0)); close(std, P.std(0))\n        return f\"ortalama tahmin std'si (belirsizlik): {std.mean():.3f}\"\n    R.test(\"mc_predict: tek çağrıda T maske, döngülü referansla aynı\", mc)\n\n    def es():\n        E = need(ns, \"EarlyStopper\", \"sınıf\")(2)\n        stops = [E.step(v) for v in [0.9, 0.7, 0.6, 0.62, 0.65]]\n        ensure(stops == [False, False, False, False, True], f\"step sonuçları {stops}\")\n        ensure(E.best_epoch == 2 and abs(E.best - 0.6) < 1e-12, f\"best {E.best}, best_epoch {E.best_epoch}\")\n        E2 = need(ns, \"EarlyStopper\", \"sınıf\")(1, min_delta=0.05)\n        ensure([E2.step(v) for v in [1.0, 0.97]] == [False, True], \"min_delta\")\n    R.test(\"EarlyStopper\", es)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def drop():\n        D = need(ns, \"MyDropout\", \"sınıf\")(0.3)\n        x = torch.ones(200_000, requires_grad=True)\n        torch.manual_seed(0)\n        y = D(x)\n        frac = (y == 0).float().mean().item()\n        ensure(abs(frac - 0.3) < 0.01, f\"sıfır oranı {frac:.3f}, 0.3 bekleniyordu\")\n        kept = y[y != 0]\n        close(kept[:5], torch.full((5,), 1 / 0.7), what=\"ölçek 1/(1−p)\")\n        y.sum().backward()\n        ensure(bool(((x.grad == 0) == (y.detach() == 0)).all()), \"düşen elemanların gradyanı 0 olmalı\")\n        D.eval()\n        z = torch.randn(10)\n        ensure(torch.equal(D(z), z), \"eval'de kimlik\")\n        return f\"sıfır oranı {frac:.4f}\"\n    R.test(\"MyDropout: oran, ölçek, gradyan, eval\", drop)\n\n    def es():\n        g = torch.Generator().manual_seed(0)\n        n = 60\n        Xtr = torch.randn(n, 10, generator=g); ytr = (Xtr[:, 0] + 0.8 * torch.randn(n, generator=g) > 0).long()\n        Xva = torch.randn(400, 10, generator=g); yva = (Xva[:, 0] + 0.8 * torch.randn(400, generator=g) > 0).long()\n        torch.manual_seed(0)\n        model = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))\n        m, best_epoch, hist = need(ns, \"fit_early_stopping\")(model, Xtr, ytr, Xva, yva, patience=10, max_epochs=500, lr=1e-2)\n        ensure(len(hist) < 500, \"aşırı uyuma giden modelde erken durmalıydı\")\n        ensure(best_epoch == int(torch.tensor(hist).argmin()), \"best_epoch doğrulama kaybının argmin'i olmalı\")\n        ensure(len(hist) - 1 - best_epoch == 10, f\"patience = 10: en iyiden sonra {len(hist) - 1 - best_epoch} epoch\")\n        m.eval()\n        with torch.no_grad():\n            v = F.cross_entropy(m(Xva), yva).item()\n        close(v, min(hist), rtol=1e-6, what=\"döndürülen modelin doğrulama kaybı = en iyi\")\n        return f\"{len(hist)} epoch'ta durdu; en iyi epoch {best_epoch}, val kaybı {min(hist):.3f} (son {hist[-1]:.3f})\"\n    R.test(\"fit_early_stopping: durur ve en iyi ağırlıkları geri yükler\", es)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    W1, b1 = torch.randn(5, 32, generator=g, dtype=torch.float64), torch.randn(32, generator=g, dtype=torch.float64) * 0.1\n    W2, b2 = torch.randn(32, 3, generator=g, dtype=torch.float64), torch.randn(3, generator=g, dtype=torch.float64) * 0.1\n    X = torch.randn(20, 5, generator=g, dtype=torch.float64)\n    masks = torch.rand(50, 20, 32, generator=g) >= 0.5\n\n    def eq():\n        out = need(ns, \"mc_logits\")(W1, b1, W2, b2, X, masks, 0.5)\n        ensure(tuple(out.shape) == (50, 20, 3), f\"shape {tuple(out.shape)}\")\n        H = torch.relu(X @ W1 + b1)\n        ref = torch.stack([(H * masks[t] / 0.5) @ W2 + b2 for t in range(50)])\n        close(out, ref)\n    R.test(\"mc_logits = T geçiş döngüsü\", eq)\n\n    def ent():\n        L = torch.randn(50, 20, 3, generator=g, dtype=torch.float64)\n        P = L.softmax(-1).mean(0)\n        close(need(ns, \"predictive_entropy\")(L), -(P * P.log()).sum(-1))\n        u = need(ns, \"predictive_entropy\")(torch.zeros(4, 2, 3, dtype=torch.float64))\n        close(u, torch.full((2,), float(torch.log(torch.tensor(3.0)))), what=\"eşit olasılık → log C\")\n    R.test(\"predictive_entropy\", ent)\n    if \"mc_logits\" not in ns:\n        return\n    W1f, b1f, W2f, b2f, Xf = W1.float(), b1.float(), W2.float(), b2.float(), torch.randn(512, 5, generator=g)\n    mk = torch.rand(100, 512, 32, generator=g) >= 0.5\n    H = torch.relu(Xf @ W1f + b1f)\n    t_loop = bench(lambda: torch.stack([(H * mk[t] / 0.5) @ W2f + b2f for t in range(100)]), repeat=3)\n    t_vec = bench(lambda: ns[\"mc_logits\"](W1f, b1f, W2f, b2f, Xf, mk, 0.5), repeat=3)\n    R.table(\"MC dropout, T = 100 geçiş, N = 512\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for t: geçiş\", fmt_time(t_loop), \"1×\"], [\"senin mc_logits\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"alpha_dropout\", \"std_dropout\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        x = rng.normal(size=400000)\n        for p in (0.1, 0.3):\n            y = need(ns, \"alpha_dropout\")(x, p, rng)\n            close((y.mean(), y.var()), (0.0, 1.0), atol=0.01, what=f\"alpha dropout p = {p}: ortalama 0, varyans 1\")\n            z = need(ns, \"std_dropout\")(x, p, rng)\n            close(z.mean(), 0.0, atol=0.01, what=\"standart dropout ortalamayı korur\")\n            ensure(z.var() > 1 + p / (1 - p) - 0.05, \"… ama varyansı 1/(1 − p) katına çıkarır\")\n        lam, al = 1.0507009873554805, 1.6732632423543772\n        selu = lambda z: lam * np.where(z > 0, z, al * (np.exp(np.minimum(z, 0)) - 1))\n        rows = []\n        for name, drop in ((\"dropout yok\", None), (\"standart dropout\", ns[\"std_dropout\"]), (\"alpha dropout\", ns[\"alpha_dropout\"])):\n            H = rng.normal(size=(2000, 128))\n            for _ in range(30):\n                H = selu(H @ (rng.normal(size=(128, 128)) / np.sqrt(128)))\n                if drop is not None:\n                    H = drop(H, 0.2, rng)\n            rows.append([name, f\"{H.mean():+.3f}\", f\"{H.std():.3f}\"])\n        ensure(abs(float(rows[2][2]) - 1) < 0.15 and abs(float(rows[1][2]) - 1) > abs(float(rows[2][2]) - 1), \"alpha dropout öz-normalizasyonu korur\")\n        R_.table(\"SELU + LeCun, 30 katman, her katmandan sonra p = 0.2 bırakma\", [\"bırakma\", \"son ortalama\", \"son std\"], rows,\n                 note=\"Standart dropout'un 0 değeri SELU için 'nötr' değildir (SELU'nun negatif doygunluğu −1.758) ve 1/q ölçeklemesi varyansı şişirir; birikince katmanlar sabit noktadan uzaklaşır. Alpha dropout bırakılan birimleri doygunluk değerine kilitler ve afin düzeltmeyle (0, 1) momentlerini korur.\")\n    R_.test(\"alpha dropout\", t)\n",
 "x2": "import numpy as np\nfrom sklearn.datasets import load_digits\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"max_norm\"))\n\n    def t():\n        W = np.array([[3.0, 0.1], [4.0, 0.2]])\n        close(need(ns, \"max_norm\")(W, 1.0), [[0.6, 0.1], [0.8, 0.2]], what=\"yalnız normu > r olan sütunlar ölçeklenir\")\n        X, y = load_digits(return_X_y=True)\n        X = X / 16\n        rng = np.random.default_rng(0)\n        p = rng.permutation(len(X))\n        tr, te = p[:300], p[300:]\n        rows = []\n        res = {}\n        for r in (None, 3.0, 1.5, 1.0):\n            W1, b1, W2, b2 = need(ns, \"train\")(X[tr], y[tr], 256, 0.5, 60, r, 0)\n            acc = lambda idx: (np.argmax(np.maximum(0, X[idx] @ W1 + b1) @ W2 + b2, 1) == y[idx]).mean()\n            res[r] = (acc(tr), acc(te), max(np.linalg.norm(W1, axis=0).max(), np.linalg.norm(W2, axis=0).max()))\n            rows.append([\"yok\" if r is None else r, f\"{res[r][0]:.3f}\", f\"{res[r][1]:.3f}\", f\"{res[r][2]:.2f}\"])\n        ensure(res[3.0][2] <= 3.0 + 1e-9 and res[1.5][2] <= 1.5 + 1e-9, \"kısıt her adımda uygulanır\")\n        ensure(res[None][2] > 3.0 and res[1.0][1] < res[None][1], \"kısıtsız normlar büyür; çok sıkı kısıt eksik uyum\")\n        R_.table(\"Rakamlar, 300 eğitim örneği, 256 gizli nöron, η = 0.5, 60 epoch\", [\"r\", \"eğitim doğr.\", \"test doğr.\", \"maks. sütun normu (W₁, W₂)\"], rows,\n                 note=\"Max-norm bir ceza değil, bir izdüşümdür: kısıt sınırına ulaşmayan nöronlara hiç dokunmaz (r = 3'te W₁ hiç etkilenmez). Bu küçük problemde etkisi sınırlı, r = 1 ise eksik uyuma yol açar. Büyük η'da bile ağırlıklar sınırsız büyüyemez — Srivastava ve ark. dropout'u yüksek öğrenme oranı + momentum + max-norm (r ≈ 3–4) ile birlikte kullanmayı önerir.\")\n    R_.test(\"max-norm\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        g = [np.array([0.9, 100.0])]\n        v = need(ns, \"clip_value\")(g, 1.0)[0]\n        n, n0 = need(ns, \"clip_global_norm\")(g, 1.0)\n        close(v, [0.9, 1.0], what=\"değere göre: yön değişir\")\n        close(n[0], np.array([0.9, 100.0]) / np.hypot(0.9, 100), what=\"norma göre: yön korunur\")\n        close(n0, np.hypot(0.9, 100), what=\"önceki norm\")\n        gs = [np.ones((2, 2)), np.ones(4) * 2]\n        cl, nn = ns[\"clip_global_norm\"](gs, 1.0)\n        close(nn, np.sqrt(4 + 16), what=\"global norm tüm parametreler üzerinden\")\n        close(np.sqrt(sum(np.sum(x * x) for x in cl)), 1.0, what=\"kırpılmış global norm = c\")\n        rng = np.random.default_rng(0)\n        rows = []\n        for scale in (0.5, 5, 50):\n            G = [rng.normal(size=20) * np.r_[np.full(10, 1.0), np.full(10, scale)]]\n            cv = ns[\"clip_value\"](G, 1.0)[0]\n            cn = ns[\"clip_global_norm\"](G, 1.0)[0][0]\n            cos = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))\n            rows.append([scale, f\"{cos(cv, G[0]):.3f}\", f\"{cos(cn, G[0]):.3f}\"])\n        ensure(float(rows[2][1]) < float(rows[0][1]) and float(rows[2][2]) > 0.999, \"değere göre kırpma büyük bileşenler baskınken yönü bozar\")\n        R_.table(\"Bileşenlerin yarısı s kat büyük bir gradyanda kırpma sonrası yön benzerliği (c = 1)\", [\"s\", \"cos(değere göre, g)\", \"cos(norma göre, g)\"], rows,\n                 note=\"Değere göre kırpma büyük bileşenleri 'düzleştirir': gradyan bir işaret vektörüne yaklaşır ve iniş yönünden sapar. Norma göre kırpma yalnız adım boyunu sınırlar. RNN'lerde (M4.3) patlayan gradyanlar genellikle tek bir yönde aniden büyür — norma göre kırpma (Pascanu ve ark.) bu yüzden standarttır; eşik, kırpmasız eğitimdeki tipik normun birkaç katı seçilir.\")\n    R_.test(\"gradyan kırpma\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_7_regularization_nn as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Üç araç, üç farklı mekanizma:

**Dropout.** Eğitimde her aktivasyonu $p$ olasılıkla sıfırla. **Inverted** sürümde kalanlar $1/(1-p)$ ile ölçeklenir; böylece $\mathbb E[y]=x$ olur ve test zamanında hiçbir şey yapılmaz:

$$
m_i\sim\text{Bernoulli}(1-p),\qquad y=\frac{x\odot m}{1-p},\qquad \frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\odot\frac{m}{1-p}.
$$

**Weight decay.** Ağırlıkları her adımda sıfıra doğru çekmek (M1.3, M3.4): $w\leftarrow w(1-\eta\lambda)$.

**Early stopping.** Doğrulama kaybını her epoch izle; `patience` epoch boyunca iyileşme yoksa dur ve **en iyi** epoch'un ağırlıklarını geri yükle. Eğitim süresini bir hiperparametreye çevirir.

Dropout'un bir yan ürünü: test zamanında da açık bırakıp $T$ kez tahmin almak (**MC dropout**) modelin belirsizliğine dair bir tahmin verir: ortalama tahmin, tahminlerin std'si, ortalama dağılımın entropisi.

## 1 · Elle

<!-- çözüm -->

Dropout'un çıktısını hesapla ve beklenen değeri kontrol et: $x_1=2$ yarı olasılıkla 4, yarı olasılıkla 0 oluyor; ortalaması 2. Ölçekleme olmasaydı ortalama 1 olurdu ve test zamanında her şeyi $1-p$ ile çarpmak gerekirdi (orijinal dropout makalesi böyle yapıyordu).

Early stopping'de kayıplar 2. epoch'tan sonra iki kez iyileşmiyor; 5. epoch'taki 0.61 asla görülmüyor. `patience` büyük seçilseydi görülürdü ama yine en iyi değil (0.6 < 0.61).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Inverted dropout, düşürme olasılığı $p=0.5$, $x=[2,4,6,8]$, maske $m=[1,0,1,1]$: $y=x\odot m/(1-p)$**

- `o1`: $y_1$
- `o2`: $y_2$
- `o4`: $y_4$
- `ex`: $\mathbb E[y_1]$ (maske rastgele)
- `test`: Test zamanı $y_1$

**Weight decay ve early stopping**

- `wd`: Ayrık decay: $w=1$, $\eta=0.1$, $\lambda=0.01$ → $w(1-\eta\lambda)$
- `best`: Doğrulama kayıpları $[0.9,0.7,0.6,0.62,0.65,0.61]$, patience 2: en iyi epoch (0-tabanlı)
- `stop`: … durdurulan epoch

In [ ]:
cevaplar = dict(
    o1=None,
    o2=None,
    o4=None,
    ex=None,
    test=None,
    wd=None,
    best=None,
    stop=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"o1":[4,0],"o2":[0,0],"o4":[16,0],"ex":[2,0],"test":[2,0],"wd":[0.999,0.00001],"best":[2,0],"stop":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, `random` serbest.

- `dropout_forward(x, p, rng, train=True)` → `(y, mask)`; her eleman için **sırayla** bir `rng.random()` çağır, `mask_i = 1 if r >= p else 0`. `train=False` ise `(list(x), None)`.
- `dropout_backward(dy, mask, p)` → `dx`.
- `early_stopping(val_losses, patience, min_delta=0.0)` → `(best_epoch, stop_epoch)`; iyileşme: `loss < best - min_delta`. Hiç durmazsa `stop_epoch` son indeks.

In [ ]:
%%writefile stage_b.py
import random


def dropout_forward(x, p, rng, train=True):
    ...


def dropout_backward(dy, mask, p):
    ...


def early_stopping(val_losses, patience, min_delta=0.0):
    ...


print(dropout_forward([2, 4, 6, 8], 0.5, random.Random(0)))
print(early_stopping([0.9, 0.7, 0.6, 0.62, 0.65, 0.61], 2))   # (2, 4)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

- `dropout_np(X, p, rng)` → `(Y, mask)`; `mask = rng.random(X.shape) >= p`.
- `mc_predict(W1, b1, W2, b2, X, p, T, rng)` → `(mean, std)`, her biri `(N, C)`. İki katmanlı ağ, dropout gizli katmanda: tüm $T$ maskeyi **tek** çağrıda çek, `rng.random((T, N, H)) >= p`; gizli aktivasyonlar `(T, N, H)`, olasılıklar softmax ile `(T, N, C)`. **Döngü yok.**
- `class EarlyStopper` — `__init__(self, patience, min_delta=0.0)`; `step(self, val_loss)` → durulacaksa `True`; özellikler `best`, `best_epoch` (0-tabanlı, `step` çağrı sayısına göre).

In [ ]:
%%writefile stage_c.py
import numpy as np


def dropout_np(X, p, rng):
    ...


def mc_predict(W1, b1, W2, b2, X, p, T, rng):
    """→ (mean, std), (N, C). Döngü yok."""
    ...


class EarlyStopper:
    def __init__(self, patience, min_delta=0.0):
        ...

    def step(self, val_loss):
        ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `MyDropout(p)` — `nn.Module`; `self.training` bayrağına göre davran (eğitim: `torch.rand_like(x) >= p` maskesi + ölçekleme; eval: kimlik).
- `fit_early_stopping(model, Xtr, ytr, Xva, yva, patience=10, max_epochs=500, lr=1e-2)` → `(model, best_epoch, history)`; tam batch `Adam`, `cross_entropy`. Her epoch: bir eğitim adımı (`model.train()`), sonra doğrulama kaybı (`model.eval()` + `torch.no_grad()`). En iyi ağırlıkları `copy.deepcopy(model.state_dict())` ile sakla, sonunda `load_state_dict` ile geri yükle. `history` doğrulama kayıpları listesi.

In [ ]:
%%writefile stage_d.py
import copy
import torch
import torch.nn.functional as F
from torch import nn


class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        ...


def fit_early_stopping(model, Xtr, ytr, Xva, yva, patience=10, max_epochs=500, lr=1e-2):
    """→ (model [en iyi ağırlıklar yüklü], best_epoch, history)"""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

MC dropout'ta $T$ geçişi döngüyle yapmak yerine $T$'yi bir batch boyutu yap (maskeler önceden üretilmiş geliyor, sonuç deterministik):

$$
H_t=\text{ReLU}(XW_1+b_1)\odot\frac{M_t}{1-p},\qquad S_t=H_tW_2+b_2,\qquad t=1..T .
$$

- `mc_logits(W1, b1, W2, b2, X, masks, p)` — `masks: (T, N, H)` bool → `(T, N, C)`. **Döngü yok.**
- `predictive_entropy(logits)` — `(T, N, C)` → `(N,)`; ortalama olasılığın entropisi $-\sum_c\bar p_c\log\bar p_c$.

In [ ]:
%%writefile stage_e.py
import torch


def mc_logits(W1, b1, W2, b2, X, masks, p):
    """masks: (T, N, H) → (T, N, C). Döngü yok."""
    ...


def predictive_entropy(logits):
    """(T, N, C) → (N,). Döngü yok."""
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Alpha dropout: SELU ile uyumlu bırakma

**Alpha dropout** (Klambauer ve ark. 2017): normal dropout bırakılan birimleri 0 yapar ve kalanları $1/q$ ile ölçekler — ortalama korunur ama varyans değişir; SELU'nun öz-normalizasyonu bozulur. Alpha dropout bırakılanları SELU'nun negatif doygunluk değerine $\alpha'=-\lambda\alpha\approx-1.758$ ayarlar, sonra ortalamayı ve varyansı geri getiren bir afin dönüşüm uygular ($q$ = tutma olasılığı):
$$a=\big(q+\alpha'^2q(1-q)\big)^{-1/2},\quad b=-a(1-q)\alpha',\quad y=a\,(x\,m+\alpha'(1-m))+b.$$

NumPy.

- `alpha_dropout(x, p, rng)` — `p` bırakma olasılığı; döngüsüz.
- `std_dropout(x, p, rng)` — inverted dropout (karşılaştırma için).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def alpha_dropout(x, p, rng):
    ...


def std_dropout(x, p, rng):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Max-norm düzenlileştirmesi

**Max-norm düzenlileştirmesi** (Srivastava ve ark. 2014; Géron): her eğitim adımından sonra her nöronun gelen ağırlık vektörünü $\|w\|_2\le r$ olacak şekilde yeniden ölçekle: $w\leftarrow w\cdot\min(1,r/\|w\|)$. Kayba ceza terimi eklemez; büyük öğrenme oranlarında patlamayı engeller, dropout'la iyi çalışır.

NumPy.

- `max_norm(W, r, axis=0)` — `W (in, out)`; her çıkış nöronunun (sütun) normunu sınırla. Döngüsüz.
- `train(X, y, hidden, lr, epochs, r, seed)` — tek gizli katmanlı (ReLU) softmax ağı, mini-batch SGD; `r=None` → kısıtsız; `r` verilirse her adımdan sonra iki ağırlık matrisine de max-norm. Eğitilmiş `(W1, b1, W2, b2)` döndür.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def max_norm(W, r, axis=0):
    ...


def train(X, y, hidden, lr, epochs, r, seed):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Gradyan kırpma: değere göre vs (global) norma göre

**Gradyan kırpma** (Géron; Pascanu ve ark. 2013): patlayan gradyanlara karşı (özellikle RNN'lerde, BN'nin zor olduğu yerlerde). İki tür:

- **değere göre** — her bileşeni $[-c,c]$'ye kırp: gradyanın **yönünü değiştirebilir** (ör. [0.9, 100] → [0.9, 1.0]);
- **norma göre** — $\|g\|>c$ ise $g\leftarrow g\,c/\|g\|$: yön korunur. Global norm: tüm parametrelerin gradyanları tek vektör sayılır (`clip_grad_norm_`).

NumPy.

- `clip_value(grads, c)`, `clip_global_norm(grads, c)` → `(kırpılmış, norm_önce)` — `grads` dizi listesi.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def clip_value(grads, c):
    ...


def clip_global_norm(grads, c):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Srivastava ve ark. (2014): Dropout

[Srivastava, Hinton, Krizhevsky, Sutskever, Salakhutdinov — Dropout: A Simple Way to Prevent Neural Networks from Overfitting (JMLR 2014)](https://jmlr.org/papers/v15/srivastava14a.html)

MNIST'te makale Tablo 2'nin küçük sürümü: 784-1024-1024-10 ağı (a) düzenlileştirmesiz, (b) L2, (c) dropout (giriş 0.2, gizli 0.5), (d) dropout + max-norm (r = 3) ile eğit. Şekil 7'deki gibi ilk katman özniteliklerini dropout'lu ve dropout'suz görselleştir; Şekil 8'deki seyreklik etkisini (gizli aktivasyonların histogramı ve ortalama aktivasyon) ölç. Ağırlık ölçekleme çıkarımını Monte Carlo ortalamasıyla (k örnek) karşılaştır (Şekil 11).


**Adımlar**

1. x2'deki max_norm'u PyTorch'a taşı (her optimizer adımından sonra torch.no_grad içinde).

**Kabul ölçütleri**

- Dropout + max-norm en düşük test hatası.
- Dropout'lu ağın gizli aktivasyonları daha seyrek.

## Makale ödevi: Gal & Ghahramani (2016): Bayesçi yaklaşım olarak dropout

[Gal & Ghahramani — Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning (ICML 2016)](https://arxiv.org/abs/1506.02142)

c aşamasındaki MC dropout'u bir regresyon problemine uygula (makale Şekil 2: CO₂ verisi yerine kendi 1B gürültülü fonksiyonun, eğitim aralığının dışına çıkan test noktalarıyla): T = 100 stokastik geçişin ortalaması ve std'si ile tahmin aralıkları çiz; dropout olasılığı ve model hassasiyeti τ ile öngörü varyansını (Denk. 8: $\tau^{-1}+\mathrm{Var}$) hesapla. Rakamlarda (MNIST/digits) döndürülmüş "1" görüntüleri için sınıf olasılıklarının belirsizliğini çiz (Şekil 4).


**Adımlar**

1. Kalibrasyon: tahmin aralıklarının kapsama oranını nominal düzeyle karşılaştır.

**Kabul ölçütleri**

- Eğitim aralığı dışında belirsizlik büyür.
- Döndürme açısı arttıkça sınıf olasılıklarının yayılımı artar.

## Kaynaklar

- [Srivastava et al. — Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html) — ©
- [CS231n — Neural Networks Part 2: Regularization (inverted dropout)](https://cs231n.github.io/neural-networks-2/#reg) — MIT
- [Gal & Ghahramani — Dropout as a Bayesian Approximation (MC dropout)](https://arxiv.org/abs/1506.02142) — arXiv
- [Dive into Deep Learning §5.6 Dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html) — CC BY-SA 4.0 / MIT-0